# ROC curves from anomaly scores

Turns the per-frame scores written by `scripts/eval_sweep.sh` into ROC curves.

A frame is scored by the reconstruction error of the clip centred on it, so scores begin
at frame `(window-1)/2` and the labels are sliced to the same offset. IPAD only.

In [ ]:
# Resolve every path through the package config, so this notebook follows whatever
# SPARSEST_DATA_ROOT / SPARSEST_OUTPUT_ROOT point at. Switch datasets here.
CONFIG_NAME = "ipad"   # or "mnist"

import sys, pathlib
REPO = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(REPO / "src"))

from sparsest.config import load_config
cfg = load_config(REPO / "configs" / f"{CONFIG_NAME}.yaml")
print("dataset:", cfg.dataset, "| window:", cfg.window)
print("outputs:", cfg.output_path("checkpoints", create=False).parent)

In [ ]:
import os, re
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

FIG_DIR = cfg.output_path("figures")
CENTER = (cfg.window - 1) // 2

# Directory written by scripts/eval_sweep.sh.
SCORE_DIR = cfg.output_path("checkpoints", create=False).parent / "scores"
print("scores from:", SCORE_DIR, "| frame offset:", CENTER)

In [ ]:
def read_scores(path):
    """Each line ends with the score: 'anomaly score for image 12 is 0.0031'."""
    scores = []
    with open(path) as handle:
        for line in handle:
            line = line.strip()
            if line.startswith("anomaly score"):
                scores.append(float(line.split()[-1]))
    return scores


def label_path_for(stem):
    """'type_1_006' -> <data_root>/mod_image/test/type_1/labels/006_label_mod.npy"""
    m = re.match(r"(type_\d+)_(\d+)$", stem)
    if not m:
        return None
    group, seq = m.groups()
    root = cfg.data_path("train").parent.parent   # .../mod_image
    return root / "test" / group / "labels" / f"{seq}_label_mod.npy"

In [ ]:
score_files = sorted(p for p in SCORE_DIR.glob("*.txt"))
assert score_files, f"no score files in {SCORE_DIR}; run scripts/eval_sweep.sh first"

all_scores, all_labels, per_sequence = [], [], {}

for path in score_files:
    scores = read_scores(path)
    lp = label_path_for(path.stem)
    if lp is None or not lp.exists():
        print(f"  {path.stem}: {len(scores)} scores, no label file -- skipped")
        continue

    labels = np.load(lp)
    # Align: scores start at CENTER and there is one per clip.
    labels = labels[CENTER:CENTER + len(scores)]
    n = min(len(scores), len(labels))
    if n < len(scores):
        print(f"  {path.stem}: {len(scores)} scores but only {len(labels)} labels; using {n}")

    per_sequence[path.stem] = (scores[:n], labels[:n])
    all_scores.extend(scores[:n]); all_labels.extend(labels[:n])
    print(f"  {path.stem}: {n} frames, {int(np.sum(labels[:n]))} anomalous")

print(f"\ntotal: {len(all_scores)} frames")

In [ ]:
fpr, tpr, _ = roc_curve(all_labels, all_scores)
roc_auc = auc(fpr, tpr)

fig, ax = plt.subplots(figsize=(5.5, 5))
ax.plot(fpr, tpr, linewidth=2, label=f"Pooled (AUC = {roc_auc:.4f})")

for name, (s, l) in per_sequence.items():
    if len(set(l.tolist())) > 1:          # a sequence with one class has no ROC
        f, t, _ = roc_curve(l, s)
        ax.plot(f, t, linewidth=0.9, alpha=0.6, label=f"{name} ({auc(f, t):.3f})")

ax.plot([0, 1], [0, 1], linestyle="--", color="gray", linewidth=1)
ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
ax.legend(fontsize=8); ax.grid(True, linestyle="--", alpha=0.4)
fig.tight_layout()
fig.savefig(FIG_DIR / "roc_curves.pdf")
print(f"pooled AUC = {roc_auc:.4f}")

In [ ]:
# Anomaly score over time for one sequence, with ground-truth anomalies shaded.
NAME = next(iter(per_sequence))
scores, labels = per_sequence[NAME]

fig, ax = plt.subplots(figsize=(10, 3.2))
frames = np.arange(CENTER, CENTER + len(scores))
ax.plot(frames, scores, linewidth=1.2)
ax.fill_between(frames, 0, max(scores), where=np.asarray(labels) > 0,
                color="red", alpha=0.2, label="Anomalous")
ax.set_xlabel("Frame"); ax.set_ylabel("Anomaly score"); ax.set_title(NAME)
ax.legend(); ax.grid(True, linestyle="--", alpha=0.4)
fig.tight_layout()
fig.savefig(FIG_DIR / f"anomaly_score_{NAME}.pdf")